# 01 · NE-US reproduction (PLAN_BRAZIL phases 1–2)
Checks the fork against the paper's NE-US setup before any Brazil work. Results go to `docs/repro_log.md`.

Data: `$DATA_ROOT/neus/` holds the Zenodo tree (record 15346612). Cell 2 downloads it if missing (1.9 GB).
Runtime: GPU for the 20-epoch runs (Colab A100/L4: about 1 h for the Transformer, 1.5 h for the GNN).

In [ ]:
# Setup. Colab: clones/updates the repo, installs requirements, mounts Drive.
# PC: start Jupyter anywhere inside the repo and set the DATA_ROOT environment variable first.
import os, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo = Path('/content/LocalizedWeatherBR')
    if repo.exists():
        subprocess.run(['git', '-C', str(repo), 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', 'https://github.com/joancsz/LocalizedWeatherBR.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/LocalizedWeather'
else:
    os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PLAN_BRAZIL.md').exists()))
    DATA_ROOT = os.environ.get('DATA_ROOT') or sys.exit('set DATA_ROOT (folder holding neus/ and brazil/) before starting Jupyter')

os.environ['DATA_ROOT'] = DATA_ROOT  # configs read ${DATA_ROOT}
PY = sys.executable                   # !{PY} runs the scripts with this kernel's Python on Colab and Windows alike
print('repo:', os.getcwd(), '| DATA_ROOT:', DATA_ROOT)

## Data

In [ ]:
!{PY} scripts/neus/download_zenodo.py --data-root "{DATA_ROOT}"

## Phase 1 · unit tests
The 2-epoch old-vs-new loss check (`configs/neus_repro.yaml`) was done once on a PC; see `docs/repro_log.md` P1.

In [ ]:
!{PY} -m pytest -q tests/

## Phase 2 · 20-epoch runs (GPU)

In [ ]:
import torch
assert torch.cuda.is_available(), 'switch to a GPU runtime'
print(torch.cuda.get_device_name(0))

In [ ]:
!{PY} Source/run.py --config configs/neus_vit_20.yaml

In [ ]:
!{PY} Source/run.py --config configs/neus_gnn_20.yaml

## Test metrics (real observations only), copy into `docs/repro_log.md`

In [ ]:
!{PY} scripts/eval/metrics.py --run-dir "{DATA_ROOT}/neus/ModelOutputs/neus_vit_20"
!{PY} scripts/eval/metrics.py --run-dir "{DATA_ROOT}/neus/ModelOutputs/neus_gnn_20"